<a href="https://colab.research.google.com/github/Yadav106/ISL/blob/main/ISL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
!kaggle datasets download -d prathumarikeri/indian-sign-language-isl

In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('/content/indian-sign-language-isl.zip','r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:

import os
import shutil
import random

base_folder = '/content/Indian'
train_folder = os.path.join(base_folder, 'train')
test_folder = os.path.join(base_folder, 'test')

if not os.path.exists(train_folder):
    os.makedirs(train_folder)

if not os.path.exists(test_folder):
    os.makedirs(test_folder)


characters = [str(i) for i in range(1, 10)] + [chr(i) for i in range(65, 91)]

for char in characters:
    char_folder = os.path.join(base_folder, char)
    train_char_folder = os.path.join(train_folder, char)
    test_char_folder = os.path.join(test_folder, char)

    if not os.path.exists(train_char_folder):
        os.makedirs(train_char_folder)

    if not os.path.exists(test_char_folder):
        os.makedirs(test_char_folder)

    files = os.listdir(char_folder)

    random.shuffle(files)

    train_files = files[:900]
    test_files = files[900:1200]

    for file in train_files:
        src = os.path.join(char_folder, file)
        dst = os.path.join(train_char_folder, file)
        shutil.move(src, dst)

    for file in test_files:
        src = os.path.join(char_folder, file)
        dst = os.path.join(test_char_folder, file)
        shutil.move(src, dst)

print("Data split completed!")


In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Convolution2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
tf.__version__

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import numpy as np

class_labels = [str(i) for i in range(1, 10)] + [chr(i) for i in range(65, 91)]

label_encoder = LabelEncoder()
label_encoder.fit(class_labels)

def one_hot_encode_labels(labels):
    integer_encoded = label_encoder.transform(labels)
    onehot_encoder = OneHotEncoder(sparse=False, categories='auto')
    integer_encoded = integer_encoded.reshape(len(integer_encoded), 1)
    onehot_encoded = onehot_encoder.fit_transform(integer_encoded)
    return onehot_encoded

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1 / 255, shear_range=0.2, zoom_range=0.2, horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1 / 255)

In [ ]:
training_set = train_datagen.flow_from_directory(
    "/content/Indian/train",
    target_size=(256, 256),
    batch_size=16,
    color_mode="grayscale",
    class_mode="categorical",
    classes=class_labels
)

test_set = test_datagen.flow_from_directory(
    "/content/Indian/test",
    target_size=(256, 256),
    batch_size=16,
    color_mode="grayscale",
    class_mode="categorical",
    classes=class_labels
)

In [ ]:
model = Sequential()

In [ ]:
model.add(Convolution2D(32, (3, 3), input_shape=(256, 256, 1), activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Convolution2D(64, (3, 3), activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Convolution2D(128, (3, 3), activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())

model.add(Dense(units=128, activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(units=64, activation="relu"))
model.add(Dropout(0.5))

model.add(Dense(units=35, activation="softmax"))


In [ ]:
model.compile(
    optimizer="Adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    training_set,
    epochs=5,
    validation_data=test_set,
)




In [ ]:
print("Training Accuracy:", max(history.history['accuracy']))
print("Validation Accuracy:", max(history.history['val_accuracy']))
print("Training Loss:", max(history.history['loss']))
print("Validation Loss:", max(history.history['val_loss']))

In [ ]:
# import numpy as np

# train_predictions = np.argmax(model.predict(training_set))
# test_predictions = np.argmax(model.predict(test_set))
# train_true_labels = np.argmax(training_set.labels)
# test_true_labels = np.argmax(test_set.labels)


# train_predictions = np.expand_dims(train_predictions)
# test_predictions = np.expand_dims(test_predictions)
# train_true_labels = np.expand_dims(train_true_labels)
# test_true_labels = np.expand_dims(test_true_labels)

# train_f1 = tf.keras.metrics.F1Score()
# train_f1.update_state(train_true_labels, train_predictions)
# train_f1_result = train_f1.result().numpy()

# test_f1 = tf.keras.metrics.F1Score()
# test_f1.update_state(test_true_labels, test_predictions)
# test_f1_result = test_f1.result().numpy()

# train_recall = tf.keras.metrics.Recall()
# train_recall.update_state(train_true_labels, train_predictions)
# train_recall_result = train_recall.result().numpy()

# test_recall = tf.keras.metrics.Recall()
# test_recall.update_state(test_true_labels, test_predictions)
# test_recall_result = test_recall.result().numpy()

# print("Training F1 Score:", train_f1_result)
# print("Training Recall:", train_recall_result)

# print("Test F1 Score:", test_f1_result)
# print("Test Recall:", test_recall_result)


In [ ]:
model.save("model.h5")

In [ ]:
model.save("model.keras")

In [ ]:
import tensorflow as tf
print(tf.__version__)